# Literature verifier audit: decision and remaining scope


Read the complete measurements in audit.ipynb and summary.csv. This review inspects the passing candidate and the identities of the extra claims it reaches; no published claims are promoted.

In [1]:
from pathlib import Path
import json, numpy as np, pandas as pd
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'starplast').is_dir() and (p / 'pyproject.toml').exists())
out = root / 'results' / 'claim_verifiers_2026_10_07'
summary = pd.read_csv(out / 'summary.csv')
heldout = pd.read_parquet(out / 'heldout.parquet')
calls = pd.read_parquet(out / 'unknown_calls.parquet')
claims = pd.read_parquet(root / 'starplast' / 'data' / 'claims.parquet')
{'combinations': len(summary), 'measured': int(summary.status.eq('measured').sum()), 'unavailable': int(summary.status.eq('layer unavailable').sum()), 'heldout_rows': len(heldout), 'unknown_calls': len(calls)}

{'combinations': 48, 'measured': 42, 'unavailable': 6, 'heldout_rows': 92354, 'unknown_calls': 234730}

## The one candidate passing the exploratory screen

Eligibility requires corrected evidence, a shared-mistake upper bound below 1.6, both calibration errors at most 0.05, at least 30 confident held-out predictions with precision at least 0.8, and some additional in-range coverage. The combined precision pools all genes reached by the recipe; it is not the precision on the two newly reached genes.

In [2]:
passing = summary[summary.eligible]
passing[['organism', 'target', 'generator', 'verifier', 'ratio', 'ratio_low', 'ratio_high', 'generator_wrong', 'agrees_n', 'agrees_rate', 'combined_calibration_error', 'combined_confident', 'combined_confident_precision', 'new_in_range']].T

,40
organism,Pf
target,pb_transferred_phenotype
generator,random_forest
verifier,literature_comention_residual
ratio,1.066
ratio_low,0.7881
ratio_high,1.331
generator_wrong,38
agrees_n,72
agrees_rate,0.6806


In [3]:
rows = []
for r in passing.itertuples():
    proposed = calls[(calls.organism == r.organism) & (calls.target == r.target) & (calls.verifier == r.verifier) & calls.prediction.notna()]
    existing = claims[(claims.organism.astype(str) == r.organism) & (claims.target.astype(str) == r.target) & claims.status.astype(str).eq('untested')]
    joined = existing.merge(proposed[['gene_id', 'prediction', 'support']], on='gene_id', validate='one_to_one')
    joined['agrees'] = joined['claim'].astype(str) == joined['prediction'].astype(str)
    rows.append(joined[['organism', 'target', 'gene_id', 'claim', 'prediction', 'agrees', 'confidence', 'lift']])
newly_reached = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
newly_reached.to_csv(out / 'newly_reached.csv', index=False)
newly_reached

,organism,target,gene_id,claim,prediction,agrees,confidence,lift
0,Pf,pb_transferred_phenotype,PF3D7_1007700,Essential,Essential,True,0.7842,1.627
1,Pf,pb_transferred_phenotype,PF3D7_1145400,Essential,Slow,False,0.7252,1.504


## Decision

No Toxoplasma candidate passes both independence and calibration. The Plasmodium phenotype candidate passes the pooled screen but adds only two previously untested in-range claims. Keep it as an exploratory candidate; do not relabel the claims as independently tested. Before promotion, measure performance conditional on this newly reached stratum and rule out source-paper reuse. The aggregate recipe precision alone does not establish either.

Plasmodium export passes dependence and calibration but reaches no unknown genes. The Plasmodium full-text graph is unavailable. Further broad coverage work should evaluate orthology or separate experimental evidence. Frozen prospective tests remain a separate open task.

In [4]:
corrected = summary[summary.corrected & summary.status.eq('measured')]
corrected[['organism', 'target', 'verifier', 'ratio_high', 'combined_calibration_error', 'new_in_range', 'outside_range_reached', 'eligible']].round(4)

,organism,target,verifier,ratio_high,combined_calibration_error,new_in_range,outside_range_reached,eligible
0,Tg,compartment,literature_comention_residual,2.721,NaN,15,17,False
2,Tg,compartment,literature_comention_ft_residual,2.044,NaN,205,155,False
4,Tg,lopit_unified,literature_comention_residual,2.264,NaN,15,17,False
6,Tg,lopit_unified,literature_comention_ft_residual,2.412,NaN,205,155,False
8,Tg,dtm_class,literature_comention_residual,2.471,NaN,0,0,False
10,Tg,dtm_class,literature_comention_ft_residual,2.613,NaN,0,0,False
12,Tg,screen_any_phenotype,literature_comention_residual,NaN,NaN,0,0,False
14,Tg,screen_any_phenotype,literature_comention_ft_residual,1.386,0.0787,0,0,False
16,Tg,cellcycle_phase,literature_comention_residual,2.639,NaN,95,18,False
18,Tg,cellcycle_phase,literature_comention_ft_residual,1.901,NaN,950,77,False


## Validation and execution record

The audit command completed successfully on the final source, and its input hashes were unchanged across the run. All 415 focused audit, claims, track-record, registry and docstring tests passed. An earlier attempt saved all measurements but failed when its last console print referenced a notebook-local variable; the CLI and clean-notebook replay tests cover that correction. The final audit was rerun in full.